# Milestone 2: Model Training and Hyperparameter Tuning

**Deadline:** `Feb 22, 2026`

1. [Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)
2. [Stochastic Gradient Descent](https://scikit-learn.org/stable/modules/sgd.html)

Tune the hyperparameters of the model.

## Import libraries

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

## Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

## Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

## Question 1

In the context of text preprocessing for NLP, which condition must be checked in addition to NaN values when assessing missingness in the comment column of train.csv?

- [ ] Presence of duplicate comments
- [ ] Presence of out-of-vocabulary tokens
- [x] Presence of empty or whitespace-only strings
- [ ] Presence of rare words

## Question 2

Convert the created_date column into datetime objects. Identify the month that occurs the most number of times in the dataset and enter the answer as a lowercase string.

In [4]:
train["created_date"] = pd.to_datetime(train["created_date"])
most_common_month = train["created_date"].dt.month.mode()[0]
pd.to_datetime(most_common_month, format="%m").strftime("%B").lower()

'may'

## Question 3

Create a new feature called total_emoticons by calculating the sum of `emoticon_1`, `emoticon_2`, and `emoticon_3` for each row. What is the maximum value observed in this new feature across the entire dataset?

In [5]:
train["total_emoticons"] = (
    train["emoticon_1"] + train["emoticon_2"] + train["emoticon_3"]
)

train["total_emoticons"].max()

np.int64(60)

## Question 4

Calculate the median character length (including spaces) of the comment column for all entries where label is equal to 3. (**Note:** Treat any missing comments as empty strings).

In [6]:
train["comment"] = train["comment"].fillna("")
train[train["label"] == 3]["comment"].str.len().median()

np.float64(128.0)

## Question 5

If you apply Min-Max Scaling to the upvote column to normalize it between the range [0, 1], what would be the resulting scaled value for an observation that originally had exactly 10 upvotes?

- [ ] 0.0345
- [ ] 0.0612
- [ ] 0.0521
- [x] 0.0498

In [7]:
scaler = MinMaxScaler()
train["upvote_scaled"] = scaler.fit_transform(train[["upvote"]])

train.loc[train["upvote"] == 10, "upvote_scaled"].iloc[0].round(4)

np.float64(0.0498)

## Question 6

What is the average word count (number of words separated by whitespace) for comments that are assigned a label of 1? (Round your answer to 2 decimal places).

In [8]:
train[train["label"] == 1]["comment"].str.split().str.len().mean().round(2)

np.float64(57.23)

## Question 7

How many comments in the entire dataset contain the substring "Trump"? The search should be case-insensitive.

In [9]:
train["comment"].str.contains("Trump", case=False, regex=False).sum()

np.int64(24398)

## Question 8

Take the comment at the very first row (index 0).

- Remove all punctuation.
- Consider the following words as stop words and remove the same :  ['a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to', 'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his', 'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your']

How many words are left in the text after these two operations?

In [10]:
comment = train.loc[0, "comment"]

# fmt: off
stop_words =  [
    'a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to',
    'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have',
    'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his',
    'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your'
]
# fmt: on

len([w for w in re.sub(r"[^\w\s]", "", comment).split() if w.lower() not in stop_words])

12

## Question 9

Convert all the text in comment column to lowercase and tokenizing using whitespace. Compute the total number of unique tokens in the dataset and enter the value.

In [11]:
train["comment"] = train["comment"].str.lower()
unique_tokens = set()
for comment in train["comment"]:
    unique_tokens.update(comment.split())

len(unique_tokens)

425413

## Question 10

Apply the TfidfVectorizer to the comment column of train.csv with stop_words as "english", min_df as 5 and ngram_range as (1,2).

How many TF-IDF features are generated using this configuration?

In [12]:
tfidf = TfidfVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2))

tfidf.fit(train["comment"])
len(tfidf.get_feature_names_out())

132974